In [1]:
import numpy as np
import pandas as pd
import glob
from helper import *
from define_model import *
from my_helpers import *
import pprint as p
from sklearn.model_selection import train_test_split
from itertools import zip_longest
from tensorflow.keras.optimizers import Adam

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
p.pprint(tf.test.is_built_with_cuda())
print('*'*50)
p.pprint(tf.config.list_physical_devices('GPU'))
print('*'*50)
p.pprint(tf.test.gpu_device_name())

True
**************************************************
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
**************************************************
'/device:GPU:0'


2023-07-27 09:54:32.812015: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-27 09:54:32.827908: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-27 09:54:32.828775: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-27 09:54:32.831853: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags

In [4]:
# Below identifiers correspond to evaluation protocol 1, 2 and 3 respectively
# identifiers - "TouchPose_crossSession", "TouchPose_crossSubjects", "TouchPose_crossGestures"
identifier_eth = "TouchPose_crossGestures"
data_path_eth = "../data/eth_data/"
data_path_chili= "../data/processed/"
model_path = "../models/"

In [5]:
files = glob.glob(data_path_chili+'*.pkl')
chili_data = pd.concat([pd.read_pickle(fp) for fp in files], ignore_index=True)
chili_data['skeleton']= chili_data['skeleton'].apply(lambda x : x.astype(np.float32))
max_skel = np.max(chili_data['skeleton'].apply(lambda x : np.max(x)))
max_cap = np.max(chili_data['cap_img'].apply(lambda x : np.max(x)))
#chili_data['skeleton']= chili_data['skeleton'].apply(lambda x : x/max_skel)
chili_data['cap_img']= chili_data['cap_img'].apply(lambda x : x/max_cap)
chili_data

,Timestamp,skeleton,ScanSizeX,ScanSizeY,trial,gesture,subject,day,cap_img
0,1.680093e+12,"[[-270.0, 288.0, 68.0], [-242.0, 276.0, 55.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
1,1.680093e+12,"[[-270.0, 288.0, 68.0], [-243.0, 276.0, 54.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
2,1.680093e+12,"[[-270.0, 288.0, 67.0], [-243.0, 276.0, 54.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
3,1.680093e+12,"[[-270.0, 288.0, 67.0], [-243.0, 276.0, 54.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
4,1.680093e+12,"[[-270.0, 288.0, 67.0], [-242.0, 276.0, 54.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
...,...,...,...,...,...,...,...,...,...
24028,1.680267e+12,"[[-280.0, 292.0, 156.0], [-265.0, 278.0, 126.0...",72,41,6,UP,13,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
24029,1.680267e+12,"[[-281.0, 293.0, 155.0], [-266.0, 280.0, 124.0...",72,41,6,UP,13,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
24030,1.680267e+12,"[[-281.0, 293.0, 154.0], [-266.0, 280.0, 123.0...",72,41,6,UP,13,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
24031,1.680267e+12,"[[-282.0, 294.0, 153.0], [-267.0, 281.0, 122.0...",72,41,6,UP,13,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."


# Load partially model

In [6]:
model_eth = get_eth_model('TouchPose_crossGestures',65)
model = create_SkeletonModel(41, 72, 1)

2023-07-27 09:53:10.882775: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-27 09:53:10.883178: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-27 09:53:10.883498: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-27 09:53:10.883853: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-27 09:53:10.884153: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from S

In [7]:
print(model_eth.input)
print(model_eth.output)

KerasTensor(type_spec=TensorSpec(shape=(None, 41, 72, 1), dtype=tf.float32, name='input_25'), name='input_25', description="created by layer 'input_25'")
KerasTensor(type_spec=TensorSpec(shape=(None, 41, 72, 1), dtype=tf.float32, name=None), name='regression_depth/BiasAdd:0', description="created by layer 'regression_depth'")


In [8]:
print(model.input)
print(model.output)

KerasTensor(type_spec=TensorSpec(shape=(None, 41, 72, 1), dtype=tf.float32, name='input_1'), name='input_1', description="created by layer 'input_1'")
KerasTensor(type_spec=TensorSpec(shape=(None, 63), dtype=tf.float32, name=None), name='regression/BiasAdd:0', description="created by layer 'regression'")


In [9]:
print(len(model.layers))

18


In [10]:
print(len(model_eth.layers))

26


In [11]:
match = []
for depth_layer, skel_layer, pos in zip(model_eth.layers,model.layers,range(len(model.layers))):
    match.append((depth_layer.name , skel_layer.name, pos ))
match

[('input_25', 'input_1', 0),
 ('conv2d_336', 'conv2d', 1),
 ('conv2d_337', 'conv2d_1', 2),
 ('max_pooling2d_72', 'max_pooling2d', 3),
 ('conv2d_338', 'conv2d_2', 4),
 ('conv2d_339', 'conv2d_3', 5),
 ('max_pooling2d_73', 'max_pooling2d_1', 6),
 ('conv2d_340', 'conv2d_4', 7),
 ('conv2d_341', 'conv2d_5', 8),
 ('max_pooling2d_74', 'max_pooling2d_2', 9),
 ('conv2d_342', 'conv2d_6', 10),
 ('conv2d_343', 'flatten', 11),
 ('up_sampling2d_72', 'dense', 12),
 ('concatenate_72', 'activation', 13),
 ('conv2d_344', 'batch_normalization', 14),
 ('conv2d_345', 'dropout', 15),
 ('up_sampling2d_73', 'dense_1', 16),
 ('concatenate_73', 'regression', 17)]

In [12]:
match = [ (''.join([ xi for xi in x[0] if not (xi.isdigit() or xi== '_')]),''.join([ xi for xi in x[1] if not (xi.isdigit() or xi== '_')]),x[2]) for x in match]
match

[('input', 'input', 0),
 ('convd', 'convd', 1),
 ('convd', 'convd', 2),
 ('maxpoolingd', 'maxpoolingd', 3),
 ('convd', 'convd', 4),
 ('convd', 'convd', 5),
 ('maxpoolingd', 'maxpoolingd', 6),
 ('convd', 'convd', 7),
 ('convd', 'convd', 8),
 ('maxpoolingd', 'maxpoolingd', 9),
 ('convd', 'convd', 10),
 ('convd', 'flatten', 11),
 ('upsamplingd', 'dense', 12),
 ('concatenate', 'activation', 13),
 ('convd', 'batchnormalization', 14),
 ('convd', 'dropout', 15),
 ('upsamplingd', 'dense', 16),
 ('concatenate', 'regression', 17)]

In [13]:
good_match = [x for x in match if x[0]==x[1]]
good_match

[('input', 'input', 0),
 ('convd', 'convd', 1),
 ('convd', 'convd', 2),
 ('maxpoolingd', 'maxpoolingd', 3),
 ('convd', 'convd', 4),
 ('convd', 'convd', 5),
 ('maxpoolingd', 'maxpoolingd', 6),
 ('convd', 'convd', 7),
 ('convd', 'convd', 8),
 ('maxpoolingd', 'maxpoolingd', 9),
 ('convd', 'convd', 10)]

In [14]:
for i in range(len(good_match)):
    pos = match[i][2]
    model.layers[pos].set_weights(model_eth.layers[pos].get_weights())

# Normalized model

In [15]:
X,y = data_splits(chili_data)

# Split data into train and test sets
X_train, X_rem, y_train, y_rem = train_test_split(X,y, test_size=0.2, random_state=42)

# Now we want the valid and test size to be equal (10% each of overall data). 
test_size = 0.5
X_valid, X_test, y_valid, y_test = train_test_split(X_rem,y_rem, test_size=0.5)

# Define custom loss function

start = 20/max_skel
stop = 50/max_skel
step = 5/max_skel

def auc_pck_norm(y_true, y_pred,start,stop,step):
    thresholds = np.arange(start, stop+step, step)  # Thresholds from 20 mm to 50 mm
    pck_values = []


    # Calculate the Euclidean distance between predicted and true joint coordinates
    distances = tf.sqrt(tf.reduce_sum(tf.square(y_true - y_pred), axis=-1))

    # Calculate the percentage of correct keypoints for each threshold
    for threshold in thresholds:
        correct_keypoints = tf.cast(distances < threshold, tf.float32)
        pck = tf.reduce_mean(correct_keypoints) * 100.0
        pck_values.append(pck)

    return tf.convert_to_tensor(pck_values, dtype=tf.float32)


optimizer = Adam(learning_rate=0.001, decay=5e-6)
# Compile model
model.compile(optimizer=optimizer, loss=custom_loss, metrics=[end_point_error, auc_pck_norm])

# Train model
history = model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=200, batch_size=32)


Epoch 1/200


2023-07-26 23:26:25.153542: I tensorflow/stream_executor/cuda/cuda_dnn.cc:368] Loaded cuDNN version 8204


601/601 [==============================] - 14s 20ms/step - loss: 3.6798 - end_point_error: 2.1920 - auc_pck: 0.0000e+00 - val_loss: 34.1445 - val_end_point_error: 1.3544 - val_auc_pck: 0.0000e+00
Epoch 2/200
601/601 [==============================] - 11s 19ms/step - loss: 0.3754 - end_point_error: 0.7581 - auc_pck: 7.4281e-04 - val_loss: 0.1356 - val_end_point_error: 0.4204 - val_auc_pck: 0.3289
Epoch 3/200
601/601 [==============================] - 11s 19ms/step - loss: 0.1927 - end_point_error: 0.5371 - auc_pck: 0.0661 - val_loss: 0.0834 - val_end_point_error: 0.3409 - val_auc_pck: 0.8400
Epoch 4/200
601/601 [==============================] - 11s 19ms/step - loss: 0.1301 - end_point_error: 0.4436 - auc_pck: 0.3226 - val_loss: 0.0667 - val_end_point_error: 0.3076 - val_auc_pck: 2.6198
Epoch 5/200
601/601 [==============================] - 11s 19ms/step - loss: 0.1010 - end_point_error: 0.3923 - auc_pck: 0.6940 - val_loss: 0.0777 - val_end_point_error: 0.2681 - val_auc_pck: 5.2573
Epoc

In [16]:
model.save(model_path+'chili_normalized_200_epochs_lr0.001_decay5e-6.h5')

# Same model as touchpose

In [15]:
X,y = data_splits(chili_data)

# Split data into train and test sets
X_train, X_rem, y_train, y_rem = train_test_split(X,y, test_size=0.2, random_state=42)

# Now since we want the valid and test size to be equal (10% each of overall data). 
# we have to define valid_size=0.5 (that is 50% of remaining data)
test_size = 0.5
X_valid, X_test, y_valid, y_test = train_test_split(X_rem,y_rem, test_size=0.5)

optimizer = Adam(learning_rate=0.001, decay=5e-6)
# Compile model
model.compile(optimizer=optimizer, loss=custom_loss, metrics=[end_point_error, auc_pck])

# Train model
history = model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=200, batch_size=32)


Epoch 1/200


2023-07-27 00:04:42.919547: I tensorflow/stream_executor/cuda/cuda_dnn.cc:368] Loaded cuDNN version 8204


601/601 [==============================] - 14s 20ms/step - loss: 276418.3125 - end_point_error: 549.3745 - auc_pck: 0.0297 - val_loss: 36331.6445 - val_end_point_error: 224.0334 - val_auc_pck: 0.1175
Epoch 2/200
601/601 [==============================] - 11s 19ms/step - loss: 51870.1055 - end_point_error: 273.5503 - auc_pck: 0.0587 - val_loss: 26749.5098 - val_end_point_error: 189.6652 - val_auc_pck: 0.2115
Epoch 3/200
601/601 [==============================] - 11s 19ms/step - loss: 36185.2344 - end_point_error: 232.2906 - auc_pck: 0.0713 - val_loss: 13773.0557 - val_end_point_error: 142.4745 - val_auc_pck: 0.2643
Epoch 4/200
601/601 [==============================] - 11s 19ms/step - loss: 25638.2461 - end_point_error: 197.5644 - auc_pck: 0.1183 - val_loss: 8191.1934 - val_end_point_error: 106.1151 - val_auc_pck: 2.1734
Epoch 5/200
601/601 [==============================] - 11s 19ms/step - loss: 27841.4473 - end_point_error: 203.3902 - auc_pck: 0.1582 - val_loss: 7914.8857 - val_end_po

In [16]:
model.save(model_path+'chili_200_epochs_lr0.001_decay5e-6.h5')

In [17]:
skeleton_err = []
chili_control = 'gesture'
chili_group = chili_data.groupby([chili_control])[["cap_img","skeleton"]]
chili_id = list(chili_group.groups.keys())

for i in range(len(chili_id)):
    test_set = [chili_id[i]]
    X_test_cap, Y_test_skeleton= data_splits(chili_group,test_set)
    Y_out_skeleton = model.predict(X_test_cap, batch_size=32)
    skeleton_error = np.mean(abs(Y_test_skeleton - Y_out_skeleton[0])) 
    skeleton_err.append(skeleton_error)
    print("Accuracy", chili_id[i],skeleton_error)


Accuracy DQ 23.094986
Accuracy DT 43.024254
Accuracy LQ 41.277657
Accuracy LT 76.68314
Accuracy UP 35.787052


# Model from Scratch

In [6]:
model = create_SkeletonModel(41, 72, 1)

2023-07-27 09:54:44.965169: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-27 09:54:44.965698: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-27 09:54:44.966123: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-27 09:54:44.966531: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-27 09:54:44.967006: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from S

In [7]:
X,y = data_splits(chili_data)

# Split data into train and test sets
X_train, X_rem, y_train, y_rem = train_test_split(X,y, test_size=0.2, random_state=42)

# Now since we want the valid and test size to be equal (10% each of overall data). 
# we have to define valid_size=0.5 (that is 50% of remaining data)
test_size = 0.5
X_valid, X_test, y_valid, y_test = train_test_split(X_rem,y_rem, test_size=0.5)

    
optimizer = Adam(learning_rate=0.001, decay=5e-6)
# Compile model
model.compile(optimizer=optimizer, loss=custom_loss, metrics=[end_point_error, auc_pck])

# Train model
history = model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=200, batch_size=32)



Epoch 1/200


2023-07-27 09:54:57.295684: I tensorflow/stream_executor/cuda/cuda_dnn.cc:368] Loaded cuDNN version 8204


601/601 [==============================] - 44s 65ms/step - loss: 282311.6875 - end_point_error: 546.2836 - auc_pck: 0.0327 - val_loss: 39054.3438 - val_end_point_error: 220.3793 - val_auc_pck: 0.1234
Epoch 2/200
601/601 [==============================] - 14s 23ms/step - loss: 51191.2617 - end_point_error: 264.5419 - auc_pck: 0.0520 - val_loss: 36608.6836 - val_end_point_error: 217.6422 - val_auc_pck: 0.0881
Epoch 3/200
601/601 [==============================] - 11s 19ms/step - loss: 47852.9375 - end_point_error: 255.4284 - auc_pck: 0.0683 - val_loss: 33197.6211 - val_end_point_error: 197.0148 - val_auc_pck: 0.0764
Epoch 4/200
601/601 [==============================] - 11s 19ms/step - loss: 41380.5117 - end_point_error: 233.4131 - auc_pck: 0.1003 - val_loss: 27957.5137 - val_end_point_error: 185.7884 - val_auc_pck: 0.7597
Epoch 5/200
601/601 [==============================] - 11s 19ms/step - loss: 36221.0742 - end_point_error: 220.3214 - auc_pck: 0.1538 - val_loss: 25770.1699 - val_end_